# Clase 1 — ¿Por qué el "dónde" importa?

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 1 — Introducción a los SIG (+ formatos de la Unidad 2) |
| **Duración** | 3 horas |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿Las escuelas primarias están donde está la población?

Es una pregunta simple de enunciar y sorprendentemente difícil de responder bien.
Para contestarla necesitamos cruzar dos cosas que viven en archivos distintos: **dónde
vive la gente** (un dato de población, por provincia) y **dónde están las escuelas**
(un dato de puntos, uno por establecimiento).

Nada de esto es exclusivo de los SIG: se podría intentar con una planilla. Lo que
agrega la dimensión espacial es la posibilidad de preguntar *dónde*, y de descubrir
que la respuesta cambia según cómo midamos.

Esa pregunta importa en ciencias sociales porque la distribución de un servicio público
en el territorio **es** una forma de desigualdad. Si hay provincias con muchas escuelas
por habitante y otras con pocas, eso no es un detalle administrativo: es parte de las
condiciones materiales de acceso a la educación.

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Explicar** qué es un SIG y por qué un mapa es un modelo del territorio, no el territorio.
2. **Distinguir** una asociación espacial de una explicación causal, usando el caso de John Snow.
3. **Cargar** datos geoespaciales en Python con GeoPandas, desde cuatro formatos distintos.
4. **Inspeccionar** una capa: cuántas filas tiene, en qué sistema de coordenadas está, qué representa cada fila.
5. **Construir** un indicador simple que combine una capa de puntos con una de polígonos.

### Lo que esta clase NO cubre

- **Cómo medir bien distancias y superficies.** Hoy vamos a tropezar deliberadamente con
  esto y lo vamos a dejar anotado. Se resuelve en la **Clase 3** (sistemas de referencia).
- **Cómo elegir la unidad territorial de análisis.** Hoy trabajamos con provincias; que esa elección cambie los resultados es el tema de la **Clase 2**.
- **Cómo hacer un mapa que comunique bien.** Los mapas de hoy son de inspección, no de
  publicación. Eso es la **Clase 5**.

## 3. Prerrequisitos

- Manejo básico de `pandas`: qué es un `DataFrame`, `head()`, filtrado por condición, `groupby()`.
- Ejecutar celdas en Google Colab.

No hace falta ningún conocimiento previo de cartografía ni de SIG.

## 3b. Material de esta clase

📽️ **Presentación Clase 1** (25 diapositivas). La teoría se expone ahí; esta notebook la
pone a prueba. Cada bloque indica a qué diapositivas corresponde.

| Bloque de la notebook | Diapositivas |
|---|---|
| Caso John Snow | 4–6 |
| GeoPandas y GeoDataFrame | 24–25 |
| Formatos de archivo | Presentación **Clase 2**, dp. 17–22 |

📖 Olaya, V. *Sistemas de Información Geográfica*, cap. 1–2 (lectura opcional).

## 4. Preparación del entorno

Fija las versiones de las librerías para que el código
haga siempre lo mismo, y descarga `sig_utils.py`, un módulo con utilidades del curso
que vamos a reutilizar en las ocho clases.

> ⏱️ Tarda entre uno y dos minutos. Ejecutala apenas abras la notebook.

In [ ]:
# Instalación (silenciosa) con versiones fijadas
!pip install -q "geopandas==1.0.1" "mapclassify==2.8.1" "folium==0.17.0" "matplotlib==3.9.2"

# Utilidades del curso
!wget -q -O sig_utils.py https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/sig_utils.py

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sig_utils import cargar, chequear_crs, resumen, CRS_ARGENTINA

print(f"GeoPandas {gpd.__version__} — entorno listo")

---

## 5. John Snow y la bomba de Broad Street

📽️ *Presentación Clase 1, diapositivas 4–6.*

### 🧭 Concepto

En 1854 un brote de cólera mató a unas 500 personas en diez días en el barrio de Golden
Square, Londres. John Snow ubicó en un plano las muertes y las bombas de agua públicas,
y encontró que los fallecidos se concentraban alrededor de la bomba de Broad Street.

Ese mapa se cita como el origen de los SIG, y con razón: es un caso temprano de usar la
posición de los hechos para razonar sobre ellos. 

### ⚠️ Lo que el mapa probó

El mapa de Snow mostró una **asociación espacial**: las muertes estaban cerca de una bomba
determinada. Eso es un patrón, y es mucho. 

Lo que inclinó la balanza no fue el mapa: fue el trabajo de campo. Snow verificó que la
cervecería del barrio, cuyos trabajadores bebían cerveza y agua de un pozo propio, casi
no tuvo muertes. Y sobre todo, se comprobó después que una cloaca filtraba a escasa
distancia de la napa que alimentaba la bomba.

> 🔍 **El punto de la clase, y del curso:** un mapa muestra dónde ocurren las cosas
> juntas. Para afirmar que una causa a la otra hace falta un mecanismo, o un diseño de
> investigación que descarte las alternativas. Vamos a volver sobre esta distinción en
> cada clase, y explícitamente en las clases 7 y 8.

---

## 6. Datos geoespaciales en Python

📽️ *Presentación Clase 1, diapositivas 24–25.*

### 🧭 Concepto

**GeoPandas** extiende `pandas` para trabajar con datos que tienen posición.

La idea es de una simplicidad útil: un `GeoDataFrame` **es** un `DataFrame` de pandas al
que se le agrega una columna especial, `geometry`, donde cada celda guarda una figura
—un punto, una línea, un polígono— en lugar de un número o un texto.

Todo lo que ya sabés de pandas sigue funcionando: `head()`, filtros, `groupby()`,
`merge()`. Lo nuevo son las operaciones que usan la columna `geometry`.

| pandas | GeoPandas |
|---|---|
| `DataFrame` | `GeoDataFrame` |
| `Series` | `GeoSeries` (la columna `geometry`) |
| `read_csv()` | `read_file()` |
| — | `.crs`, `.plot()`, `.explore()`, `.area`, `.distance()` |

### ▶️ Ejecución

Cargamos las provincias argentinas. La función `cargar()` del módulo del curso descarga
el archivo la primera vez, lo guarda en caché y lo abre declarando su sistema de
coordenadas — así ninguna notebook tiene que adivinarlo.

In [ ]:
provincias = cargar("provincias")
provincias.head()

Fijate en la última columna: `geometry`. Cada fila tiene un polígono con el contorno de
esa provincia. Es la única diferencia con una planilla común.

In [ ]:
# ¿Qué tipo de objeto es cada cosa?
print("El objeto completo:   ", type(provincias))
print("La columna geometry:  ", type(provincias.geometry))
print("Una geometría suelta: ", type(provincias.geometry.iloc[0]))
print()
print("Columnas disponibles:", list(provincias.columns))

### ✅ Comprobación

Antes de interpretar cualquier cosa, tres preguntas a los datos: **¿cuántas filas hay?**,
**¿en qué sistema de coordenadas están?**, **¿hay geometrías vacías o nulas?**

In [ ]:
resumen(provincias, columnas=["poblacion", "total_hogares", "superficie_km2"],
        nombre="Provincias de Argentina")

# Argentina tiene 23 provincias más la Ciudad Autónoma de Buenos Aires
assert len(provincias) == 24, f"Se esperaban 24 jurisdicciones, hay {len(provincias)}"
print("\n✓ Las 24 jurisdicciones están presentes")

### ▶️ Un primer vistazo

`.plot()` dibuja la columna `geometry` con matplotlib. Es el equivalente de `head()`
para la parte espacial: sirve para darse cuenta rápido de que algo está mal.

In [ ]:
provincias.plot(figsize=(6, 9), edgecolor="white", linewidth=0.5, color="#8da0cb")
plt.title("Provincias de Argentina")
plt.axis("off")
plt.show()

### 🔍 Interpretación

Reconocemos el mapa, así que las geometrías son plausibles. Pero notá dos cosas:

1. **Los ejes están en grados**, no en kilómetros. Vamos de −73 a −53 en el eje X. Esas
   son coordenadas geográficas (longitud y latitud), no metros.
2. **El país se ve "achatado"** respecto de cómo suele dibujarse. Estamos representando
   una superficie curva sobre una pantalla plana sin haber elegido cómo hacerlo.

Las dos observaciones son la misma cosa, y son el tema de la **Clase 3**.

---

## 7. Formatos de archivo

📽️ *Presentación **Clase 2**, diapositivas 17–22.*

### 🧭 Concepto

Los datos geoespaciales circulan en varios formatos. La buena noticia es que GeoPandas
los abre casi todos con **la misma función**: `gpd.read_file()`. Lo que cambia no es cómo
se leen, sino qué garantías da cada uno.

| Formato | Archivos | Fortaleza | Límite importante |
|---|---|---|---|
| **GeoPackage** (`.gpkg`) | 1 | Abierto, un solo archivo, varias capas, nombres largos | Menos difundido de lo que merece |
| **GeoJSON** (`.geojson`) | 1 | Texto plano, legible, nativo de la web | Pesado; solo EPSG:4326 |
| **Shapefile** (`.shp`) | **3 como mínimo** | Estándar de facto, todo lo lee | Nombres de campo de **10 caracteres**; una sola geometría por archivo |
| **CSV con coordenadas** | 1 | Sale de cualquier sistema no geográfico | **No guarda el CRS**: hay que saberlo aparte |

> ⚠️ El Shapefile es el más usado y el más problemático. Vamos a verlo en un caso real.

### ▶️ Ejecución — la misma función para todo

Ya cargamos un GeoPackage (las provincias). Ahora una línea y un conjunto de puntos.

In [ ]:
ruta40   = cargar("ruta40")     # GeoPackage con una línea
escuelas = cargar("escuelas")   # GeoPackage con 22.753 puntos

In [ ]:
# Cada capa tiene un tipo de geometría distinto
for nombre, capa in [("provincias", provincias), ("ruta 40", ruta40), ("escuelas", escuelas)]:
    print(f"{nombre:12} {str(capa.geom_type.unique()):22} {len(capa):>6} filas")

### ⚠️ El caso del Shapefile: nombres truncados y acentos rotos

Los datos de provincias que estamos usando vinieron originalmente en Shapefile. Así se
llamaban las columnas en ese archivo:

| Nombre real | Nombre en el Shapefile |
|---|---|
| `poblacion` | `poblacion_` |
| `total_hogares` | `total_hoga` |

El formato corta los nombres a diez caracteres, sin avisar. `total_hoga` no significa
nada; alguien tiene que recordar qué era. Y si el archivo tuviera `poblacion_2010` y
`poblacion_2022`, **las dos se llamarían `poblacion_`** y una pisaría a la otra.

Además, el archivo original no declaraba su codificación de caracteres, de modo que
"Córdoba" se leía como "CÃ³rdoba" según con qué programa se abriera.

Los datos que usamos hoy ya vienen corregidos en GeoPackage. Es exactamente por esto que
**en este curso exportamos siempre a `.gpkg`**.

### ▶️ El caso del CSV: coordenadas sin sistema de referencia

Es el formato más común cuando los datos salen de un sistema que no es geográfico: un
padrón, una encuesta, un registro administrativo. Tenemos dos columnas con números y
nada más.

Construir la geometría a mano es un buen ejercicio para ver qué hace `read_file()` por
detrás.

In [ ]:
# Simulamos la situación: partimos de una tabla común, sin geometría
tabla = escuelas[["establecimiento", "provincia", "latitud", "longitud"]].head(500).copy()
print("Es un DataFrame de pandas:", type(tabla).__name__)
tabla.head(3)

In [ ]:
# points_from_xy espera (X, Y) = (longitud, latitud). En ese orden.
capa_escuelas = gpd.GeoDataFrame(
    tabla,
    geometry=gpd.points_from_xy(tabla["longitud"], tabla["latitud"]),
    crs="EPSG:4326",          # <- este dato NO estaba en el CSV: lo aportamos nosotros
)
print("Ahora es un GeoDataFrame:", type(capa_escuelas).__name__)
capa_escuelas.head(3)

> ⚠️ **El orden importa y es contraintuitivo.** Decimos "latitud y longitud", pero
> `points_from_xy` pide **(longitud, latitud)**, porque en un plano X va antes que Y.
> Invertirlos es el error más frecuente de la clase: las escuelas argentinas aparecerían
> en China. Lo comprobamos abajo.

> ⚠️ **El CRS lo pusimos nosotros.** El CSV traía dos columnas de números. Que sean
> grados WGS 84 es información externa al archivo, que viene de la documentación de la
> fuente. Si nos equivocamos acá, todo lo que sigue está mal y nada nos avisa.

### ✅ Comprobación — ¿los puntos caen donde deberían?

In [ ]:
# Test barato y efectivo: ¿las coordenadas están dentro del territorio argentino?
minx, miny, maxx, maxy = capa_escuelas.total_bounds
print(f"Longitud: {minx:.2f} a {maxx:.2f}   (Argentina va de -73,6 a -53,6)")
print(f"Latitud:  {miny:.2f} a {maxy:.2f}   (Argentina va de -55,1 a -21,8)")

en_argentina = (-75 < minx) and (maxx < -52) and (-56 < miny) and (maxy < -21)
print("\n✓ Las coordenadas caen dentro de Argentina" if en_argentina
      else "\n✗ Algo está mal: ¿invertiste latitud y longitud?")

In [ ]:
# Qué pasaría si las invirtiéramos (no ejecutes esto en tu análisis real)
invertidas = gpd.points_from_xy(tabla["latitud"], tabla["longitud"])
print(f"Invertidas, el primer punto queda en: {invertidas[0]}")
print("Eso es el desierto de Taklamakán, en China.")

---

## 8. 🧪 Actividad integradora — ¿dónde faltan escuelas?

Volvemos a la pregunta del principio. Ahora tenemos todo para responderla.

### 🧭 Concepto

Tenemos escuelas (puntos) y provincias (polígonos con población). Para relacionarlos
necesitamos saber **en qué provincia cae cada escuela**. Esa operación —asignar a cada
punto el polígono que lo contiene— se llama **unión espacial** (*spatial join*), y es
una de las operaciones fundamentales de un SIG. La vamos a estudiar en profundidad en
la Clase 6; hoy la usamos como caja negra.

In [ ]:
# Contamos escuelas por provincia usando la columna que ya trae el padrón
escuelas_por_provincia = (
    escuelas.groupby("provincia").size().reset_index(name="escuelas")
)

# Unimos con la población
tabla_final = provincias[["provincia", "poblacion"]].merge(
    escuelas_por_provincia, on="provincia", how="left"
)
tabla_final["escuelas"] = tabla_final["escuelas"].fillna(0).astype(int)

print(f"Provincias sin coincidencia de nombre: {(tabla_final.escuelas == 0).sum()}")
tabla_final.head()

### ✅ Comprobación — cuidado con los nombres

Cruzar tablas por nombre de provincia es frágil: "Tierra del Fuego" puede aparecer con o
sin "Antártida e Islas del Atlántico Sur". Antes de seguir, verificamos que no perdimos
escuelas en el camino.

In [ ]:
total_padron = len(escuelas)
total_cruce  = tabla_final["escuelas"].sum()
print(f"Escuelas en el padrón:   {total_padron}")
print(f"Escuelas tras el cruce:  {total_cruce}")
print(f"Perdidas en el cruce:    {total_padron - total_cruce}")

if total_padron != total_cruce:
    faltan = set(escuelas["provincia"]) - set(provincias["provincia"])
    print(f"\nNombres que no coinciden: {faltan}")
else:
    print("\n✓ No se perdió ninguna escuela")

### ▶️ Conteo contra tasa

Ahora la parte interesante. Calculamos las dos cosas y las comparamos.

In [ ]:
tabla_final["escuelas_por_10mil_hab"] = (
    tabla_final["escuelas"] / tabla_final["poblacion"] * 10_000
)

print("=== Más escuelas en términos absolutos ===")
print(tabla_final.nlargest(5, "escuelas")[["provincia", "escuelas", "escuelas_por_10mil_hab"]]
      .to_string(index=False))

print("\n=== Más escuelas por cada 10.000 habitantes ===")
print(tabla_final.nlargest(5, "escuelas_por_10mil_hab")[["provincia", "escuelas", "escuelas_por_10mil_hab"]]
      .to_string(index=False))

print("\n=== Menos escuelas por cada 10.000 habitantes ===")
print(tabla_final.nsmallest(5, "escuelas_por_10mil_hab")[["provincia", "escuelas", "escuelas_por_10mil_hab"]]
      .to_string(index=False))

### 🔍 Interpretación — y la trampa

Las dos listas **no se parecen**. Buenos Aires encabeza el conteo absoluto y queda entre
las últimas por habitante. Las provincias con más escuelas por habitante son, en general,
las menos pobladas.

Antes de concluir que "en Buenos Aires faltan escuelas", tres advertencias:

1. **Una escuela no es una unidad de servicio comparable.** Una escuela rural de 20
   alumnos y una urbana de 900 cuentan igual en este indicador. Si tuviéramos matrícula
   por establecimiento —y la tenemos— el indicador razonable sería alumnos por escuela,
   no escuelas por habitante.

2. **La población incluye a todas las edades.** El denominador correcto sería la
   población en edad escolar, no la total. Estamos comparando escuelas primarias contra
   jubilados, entre otros.

3. **La provincia es una unidad enorme y arbitraria** para esta pregunta. Dentro de
   Buenos Aires conviven el conurbano y el sudoeste despoblado. Un promedio provincial
   los mezcla y no describe bien a ninguno de los dos. **Esto es el problema de la
   unidad de área modificable, y es el tema de la próxima clase.**

> Lo que sí podemos afirmar: la relación entre escuelas y población **no es uniforme en
> el territorio**, y la magnitud de la diferencia depende de cómo se mida. Eso es un
> hallazgo real, y es todo lo que estos datos sostienen hoy.

### ▶️ Y ahora sí, un mapa

In [ ]:
mapa = provincias.merge(
    tabla_final[["provincia", "escuelas", "escuelas_por_10mil_hab"]],
    on="provincia", how="left",
)

fig, ejes = plt.subplots(1, 2, figsize=(12, 8))
mapa.plot(column="escuelas", cmap="OrRd", legend=True, ax=ejes[0],
          edgecolor="white", linewidth=0.4)
ejes[0].set_title("Escuelas (conteo absoluto)")
mapa.plot(column="escuelas_por_10mil_hab", cmap="OrRd", legend=True, ax=ejes[1],
          edgecolor="white", linewidth=0.4)
ejes[1].set_title("Escuelas por 10.000 habitantes")
for eje in ejes:
    eje.set_axis_off()
plt.tight_layout()
plt.show()

> 🔍 **Son dos mapas del mismo dato y cuentan historias opuestas.** El de la izquierda
> dice "las escuelas están en el centro del país"; el de la derecha, "las escuelas están
> en el sur y el noroeste". Ninguno miente. Responden preguntas distintas.
>
> Elegir cuál mostrar es una decisión con consecuencias, y es un tema de la Clase 5.

---

## 9. 🤖 Actividad con IA generativa — la plausibilidad no es evidencia

Vas a usar un asistente (ChatGPT, Claude, Gemini, el que uses) para algo que **no le
diste**: describir un archivo que no vio.

### Consigna

1. Pedile esto, tal cual, **sin adjuntar ningún archivo**:

   > Tengo un archivo llamado `escuelas_primarias.gpkg` con datos de escuelas primarias
   > de Argentina. Decime qué columnas tiene, cuántas filas y en qué sistema de
   > coordenadas está.

2. Pegá la respuesta completa en la celda de abajo, sin editarla.
3. Ejecutá la celda de verificación.
4. Completá el veredicto.

### ¿Por qué esta actividad?

El asistente no tiene forma de saber qué hay en ese archivo. Pero casi con seguridad va a
responder con una lista concreta de columnas, un número de filas y un código EPSG, en un
tono seguro. Ese es el punto: **una respuesta puede ser plausible, estar bien formateada
y ser inventada**. La única forma de saberlo es mirar el dato.

In [ ]:
# 1. Pegá acá la respuesta del asistente, tal cual la recibiste
respuesta_ia = """
(pegar acá)
"""

# 2. Verificación contra el archivo real
print("=== LO QUE DICE EL ARCHIVO ===")
print(f"Filas: {len(escuelas)}")
print(f"CRS:   {escuelas.crs.to_string()}")
print(f"Columnas ({len(escuelas.columns)}):")
for c in escuelas.columns:
    print(f"   - {c}")

### Veredicto

Completá, en tus palabras:

| Afirmación del asistente | ¿Coincide? | Evidencia |
|---|---|---|
| Columnas | | |
| Cantidad de filas | | |
| Sistema de coordenadas | | |

**¿El asistente aclaró que no podía saberlo?** ▸ *(sí / no)*

**En una frase: ¿qué te llevás de esto?** ▸

---

## 10. Cierre

### Glosario de la clase

| Término | Definición |
|---|---|
| **SIG** | Sistema que integra datos, métodos, tecnología y personas para capturar, analizar y representar información georreferenciada. No es solo un software. |
| **Capa** | Conjunto de elementos geográficos del mismo tipo temático. Se superponen para modelar el territorio. |
| **GeoDataFrame** | `DataFrame` de pandas con una columna `geometry`. |
| **GeoSeries** | La columna de geometrías de un `GeoDataFrame`. |
| **CRS** | Sistema de referencia de coordenadas: la convención que da sentido a un par de números. Clase 3. |
| **Unión espacial** | Cruce de dos capas según su relación en el espacio, no por una columna en común. Clase 6. |
| **Asociación espacial** | Dos fenómenos aparecen juntos en el territorio. **No implica** que uno cause al otro. |

### Autoevaluación

1. ¿Por qué el mapa de Snow, por sí solo, no demuestra que el agua causaba el cólera?
2. Tenés un CSV con columnas `lat` y `lon`. ¿Qué información necesitás que **no** está en
   el archivo para convertirlo en una capa geográfica?
3. Los dos mapas de escuelas dicen cosas distintas. ¿Cuál usarías para decidir dónde
   construir una escuela nueva, y por qué?

### 📦 Material opcional

- `Clase_1_opcional.ipynb`: historia de los SIG, el modelo ráster en detalle, y el mapa
  original de Snow reconstruido con datos reales de 1854.

### Tarea para la próxima clase

Elegí **una pregunta territorial** de tu propio campo de investigación —algo del tipo
"¿dónde ocurre X?" o "¿X e Y ocurren en los mismos lugares?"— y escribí en tres o cuatro
oraciones:

- cuál es la pregunta;
- qué dos capas de datos necesitarías para responderla;
- cuál sería la unidad territorial de análisis (provincia, departamento, radio censal,
  barrio) y por qué.

La traemos a la Clase 2, donde vamos a ver que esa última decisión es mucho menos
inocente de lo que parece.

---

### Y lo que quedó pendiente

Dos cosas que hicimos mal a propósito y vamos a arreglar:

- **Medimos en grados.** Los ejes del mapa, la extensión de las capas: todo en grados.
  Para medir distancias y superficies eso no sirve. → **Clase 3**
- **Usamos la provincia como unidad de análisis** sin justificarlo. → **Clase 2**